# 01 - Extracción de datos
## Contratos Menores - Ayuntamiento de Vilagarcía de Arousa

El primer paso del proyecto es descargar el dataset y entender qué contiene.

**Fuente:** https://sede.vilagarcia.gal/dcsv/C0G6N1Z26P99PL6H

## Paso 0 — Instalar dependencias
Ejecuta esta celda solo la primera vez. Después reinicia el kernel (Kernel → Restart).

In [1]:
%pip install --prefer-binary -q -r ../requirements.txt

Note: you may need to restart the kernel to use updated packages.


## Paso 1 — Descargar el fichero Excel

Descargamos el dataset desde la sede electrónica y lo guardamos en `data/raw/`.  
Si ya existe no lo volvemos a descargar (idempotencia básica).

In [4]:
import requests
from pathlib import Path

URL     = "https://sede.vilagarcia.gal/dcsv/C0G6N1Z26P99PL6H"
DESTINO = Path("../data/raw/contratos_menores.xlsx")

# Comprobamos si el fichero ya existe para no descargarlo de nuevo (idempotencia)
if DESTINO.exists():
    print(f"El fichero ya existe: {DESTINO}")
else:
    print("Descargando...")
    # Petición HTTP GET a la sede electrónica del ayuntamiento
    respuesta = requests.get(URL, timeout=30)
    # Si el servidor devuelve un error (4xx, 5xx) lanza una excepción aquí y paramos
    respuesta.raise_for_status()
    # Escribimos el contenido binario del fichero descargado en disco
    DESTINO.write_bytes(respuesta.content)
    print(f"Guardado en: {DESTINO}  ({len(respuesta.content):,} bytes)")

Descargando...
Guardado en: ../data/raw/contratos_menores.xlsx  (125,147 bytes)


## Paso 2 — Cargar el Excel con pandas y ver su estructura

Antes de tocar nada, entendemos qué hay dentro.

In [5]:
import pandas as pd

# Cargamos el Excel en un DataFrame de pandas para inspeccionarlo
df = pd.read_excel(DESTINO)

# Dimensiones del dataset
print(f"Filas:    {df.shape[0]}")
print(f"Columnas: {df.shape[1]}")

# Listamos los nombres de columna tal como vienen en el fichero original
print(f"\nNombres de columnas:")
for col in df.columns:
    print(f"  - {col}")

# Contamos nulos por columna — ayuda a detectar campos problemáticos antes de limpiar
print(f"\nValores vacíos:")
print(df.isnull().sum())

print(f"\nPrimeras filas:")
print(df.head())

Filas:    631
Columnas: 20

Nombres de columnas:
  - Unnamed: 0
  - Entidad Contratante
  - Número de Referencia del Contrato
  - Objeto del Contrato
  - Valor estimado del contrato (en euros)
  - Presupuesto base sin impuestos (en euros)
  - Plazo ejecución (en meses)
  - Código CPV del objeto del contrato
  - Tipo de contratación
  - Tipo de Contrato
  - Tipo de procedimiento
  - Sistema de contratación
  - Tramitación
  - Fecha formalización
  - Importe total ofertado (sin impuestos) (en euros)
  - Importe total ofertado (con impuestos) (en euros)
  - URL a la licitación específica del expediente
  - Observaciones
  - Contratistas
  - Lista de lotes

Valores vacíos:
Unnamed: 0                                           631
Entidad Contratante                                    0
Número de Referencia del Contrato                      0
Objeto del Contrato                                    0
Valor estimado del contrato (en euros)                 3
Presupuesto base sin impuestos (en eu

## Paso 3 — Primera revisión de calidad

Cuántos valores vacíos hay en cada columna y qué tipo de dato tiene cada una.

In [11]:
# Construimos una tabla resumen con el tipo de dato, nulos y porcentaje de nulos de cada columna
resumen = pd.DataFrame({
    "tipo":     df.dtypes,                           # tipo inferido por pandas (str, float64...)
    "nulos":    df.isnull().sum(),                   # cantidad de valores vacíos
    "% nulos":  (df.isnull().mean() * 100).round(1) # porcentaje sobre el total de filas
})
resumen

,tipo,nulos,% nulos
Unnamed: 0,float64,631,100.0
Entidad Contratante,str,0,0.0
Número de Referencia del Contrato,str,0,0.0
Objeto del Contrato,str,0,0.0
Valor estimado del contrato (en euros),str,3,0.5
Presupuesto base sin impuestos (en euros),str,0,0.0
Plazo ejecución (en meses),float64,0,0.0
Código CPV del objeto del contrato,str,607,96.2
Tipo de contratación,str,0,0.0
Tipo de Contrato,str,0,0.0
